# Step 13 - Gender-bias detector retrained on PESO (Layer A/B silver labels)

**What is trained.** Three small models on the 7,971 PESO units:
1. **Detection:** TF-IDF + logistic regression, biased vs not biased.
2. **Explicit / implicit classification:** TF-IDF + logistic regression, three classes `none`, `implicit`, `explicit`.
3. **Span localisation:** a token tagger (logistic regression on word and context features) that marks the words carrying the bias.

**Labels.** The Step 8c rule pre-annotator (Layer A then Layer B). They are silver labels, **not human labels**. Therefore every score below measures how well the models reproduce those rules, not whether the labels are correct. Units labelled `unclear` (691: coded wording, "Female/Male" listings, ambiguous gender mentions) are **excluded from training and scoring** because the schema leaves them to a human; the model's output on them is shown separately.

**Evaluation.** 5-fold GroupKFold by employer (`employer_id`), seed 42, so no employer is in both train and test. 95% intervals resample whole employers 1,000 times. Sensitivity runs: keywords removed, and distinct texts only. A rule-based model would score 1.0 by construction and is not reported.

**Span metrics.** Gold span = the span of the pre-annotator. *Exact match*: predicted (start, end) equals gold. *Overlap*: a predicted span matches when intersection-over-union is at least 0.5. *Token overlap F1*: F1 over the word tokens covered. Reported (a) given a gold-biased unit and (b) end to end: the tagger only runs where the detection model says biased.

In [1]:
import json, re, platform
from pathlib import Path
import numpy as np, pandas as pd, sklearn, joblib
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold

SEED, N_SPLITS, N_BOOT = 42, 5, 1000
OUT = Path("outputs/equal_step13_gender_detector"); OUT.mkdir(parents=True, exist_ok=True)
json.dump(dict(python=platform.python_version(), platform=platform.platform(), numpy=np.__version__, pandas=pd.__version__,
               scikit_learn=sklearn.__version__, seed=SEED, cv="GroupKFold(5) by employer_id", bootstrap=f"{N_BOOT} resamples of employers",
               text_model="TfidfVectorizer(1-2 grams, min_df=2, sublinear_tf) + LogisticRegression(class_weight=balanced, max_iter=2000)",
               tagger="DictVectorizer on word/prefix/suffix/shape/neighbour features + LogisticRegression(class_weight=balanced), threshold 0.5"),
          open(OUT / "step13_environment.json", "w"), indent=2)

## 1. Silver labels and gold spans (pre-annotator copied unchanged from Step 8c)

In [2]:
MASC = ("active adventurous aggress ambitio analy assert athlet autonom boast challeng compet confident "
        "courag decisive decision determin dominan driven fearless force headstrong hierarch impulsive "
        "independen individual intellect lead logic objective opinion outspoken persist self-relian "
        "self-sufficien stubborn superior unreasonab").split()
FEM = ("agree affectionate child cheer collab commit communal compassion connect considerate cooperat "
       "depend emotiona empath feel flatterable gentle honest interpersonal interdependen kind kinship "
       "loyal modest nurtur pleasant polite quiet respons sensitiv submissive support sympath tender "
       "together trust understand warm enthusias inclusive yield share").split()
FP = ("confidential", "committee", "shareholder", "kinds")
DUTY = ("analy", "lead", "respons", "support", "decision", "challeng", "collab", "cooperat")

def coded_hits(text, stems):
    out = []
    for m in re.finditer(r"[a-z][a-z-]*", text.lower()):
        w = m.group()
        if w.startswith(FP): continue
        ps = [w] + (w.split("-") if "-" in w else [])
        if any(p.startswith(s) for p in ps for s in stems): out.append((m.start(), m.end(), w))
    return out

MALE = {"std": r"male|males|men|gentlemen", "lem": r"male|males|men|man|gentlemen|gentleman"}
FEMALE = {"std": r"female|females|women|ladies", "lem": r"female|females|women|woman|ladies|lady"}
INCL_RE = re.compile(r"\b(?:either\s+gender|any\s+gender|regardless\s+of\s+(?:sex|gender)|men\s+and\s+women|"
                     r"gender\s*[:\-]?\s*(?:any|open|not\s+required))\b", re.I)
TITLE_RE = re.compile(r"\b(?:salesman|salesmen|saleslady|salesgirl|waiter|waitress|chairman|foreman|watchman|cameraman|busboy|houseboy|stewardess)\b", re.I)
PRON_BIN_RE = re.compile(r"\b(?:he\s*/\s*she|she\s*/\s*he|he\s+or\s+she|his\s*/\s*her|s/he)\b", re.I)
PRON_RE = re.compile(r"\b(?:he|she|his|her|him)\b", re.I)
ONLY_RE = re.compile(r"\b(?:only|exclusively|must\s+be|should\s+be|required)\b", re.I)
PREF_RE = re.compile(r"\b(?:prefer|preferably|preferred|preference)\b", re.I)

def layerA(text, lemma=False):
    k = "lem" if lemma else "std"
    m_re, f_re = re.compile(r"\b(?:%s)\b" % MALE[k], re.I), re.compile(r"\b(?:%s)\b" % FEMALE[k], re.I)
    m, f = m_re.search(text), f_re.search(text)
    n_words = len(re.findall(r"\w+", text))
    out = dict(gender_reference="no", gender_expression_type="none", gender_direction="na", gender_coding="na", span=None, context_dependent="no")
    inc, pb = INCL_RE.search(text), PRON_BIN_RE.search(text)
    if inc:
        out.update(gender_reference="yes", gender_expression_type="inclusive", gender_direction="multiple", span=inc.span())
    elif m and f:
        s = min(m.start(), f.start()); e = max(m.end(), f.end())
        out.update(gender_reference="yes", gender_expression_type="binary_listing", gender_direction="multiple", span=(s, e))
    elif m or f:
        h = m or f; d = "male" if m else "female"
        t = "restriction" if (n_words <= 4 or ONLY_RE.search(text)) and not PREF_RE.search(text) else ("preference" if PREF_RE.search(text) else "unclear")
        out.update(gender_reference="yes", gender_expression_type=t, gender_direction=d, span=h.span())
    elif pb:
        out.update(gender_reference="yes", gender_expression_type="binary_listing", gender_direction="multiple", span=pb.span())
    elif TITLE_RE.search(text) or PRON_RE.search(text):
        h = TITLE_RE.search(text) or PRON_RE.search(text)
        w = h.group().lower(); d = "male" if w in ("salesman", "salesmen", "waiter", "chairman", "foreman", "watchman", "cameraman", "busboy", "houseboy", "he", "his", "him") else "female"
        out.update(gender_reference="yes", gender_expression_type="gendered_title_pronoun", gender_direction=d, span=h.span())
    else:
        mh, fh = coded_hits(text, MASC), coded_hits(text, FEM)
        if mh or fh:
            allh = sorted(mh + fh)
            out.update(gender_expression_type="gender_coded", gender_coding="agentic" if mh and not fh else "communal" if fh and not mh else "mixed",
                       span=(allh[0][0], allh[0][1]), context_dependent="yes" if all(w.startswith(DUTY) for *_, w in allh) else "no")
    return out

BMAP = {  # Layer A type -> (bias_type, bias_subtype, rewrite_action)
    "restriction": ("explicit", "gender_exclusion", "required"), "preference": ("explicit", "gender_preference", "required"),
    "binary_listing": ("unclear", "unclear", "recommended"), "gendered_title_pronoun": ("implicit", "gendered_role_language", "recommended"),
    "gender_coded": ("unclear", "unclear", "manual_review"), "inclusive": ("none", "none", "not_needed"), "none": ("none", "none", "not_needed"),
    "unclear": ("unclear", "unclear", "manual_review")}
PRESENCE = {"explicit": "biased", "implicit": "biased", "none": "not_biased", "unclear": "unclear"}

def preannotate(text, lemma=False):
    a = layerA(text, lemma)
    bt, st, ra = BMAP[a["gender_expression_type"]]
    rewrite = "Open to all qualified candidates" if (bt == "explicit" or a["gender_expression_type"] == "binary_listing") and len(re.findall(r"\w+", text)) <= 4 else ""
    return dict(**a, bias_type=bt, bias_presence=PRESENCE[bt], bias_subtype=st, rewrite_action=ra, suggested_rewrite=rewrite)


In [3]:
units = pd.read_csv("outputs/equal_step4_sentence_embeddings/step4_sentence_units.csv", encoding="utf-8-sig", dtype=str, keep_default_na=False)
rows = []
for s in units.sentence:
    r = preannotate(s); rows.append(dict(bias_type=r["bias_type"], layerA=r["gender_expression_type"], span=r["span"]))
d = pd.concat([units[["sentence_id", "record_id", "employer_id", "sentence"]], pd.DataFrame(rows)], axis=1)
print(d.bias_type.value_counts().to_string())
ds = d[d.bias_type != "unclear"].reset_index(drop=True)
CL = ["none", "implicit", "explicit"]
ds["y3"] = ds.bias_type.map({c: i for i, c in enumerate(CL)}); ds["yb"] = (ds.y3 > 0).astype(int)
print("\nin scope (unclear excluded):", len(ds), "| employers:", ds.employer_id.nunique())
summ = ds.groupby("bias_type").agg(units=("sentence", "size"), distinct_texts=("sentence", "nunique"), employers=("employer_id", "nunique")).loc[CL]
display(summ); summ.to_csv(OUT / "step13_label_counts.csv")
print("Layer A types behind implicit:", ds[ds.bias_type == "implicit"].layerA.value_counts().to_dict())
print("implicit distinct texts:", sorted(ds[ds.bias_type == "implicit"].sentence.str.lower().unique())[:15])

bias_type
none        6929
unclear      691
explicit     313
implicit      38

in scope (unclear excluded): 7280 | employers: 320


,units,distinct_texts,employers
bias_type,,,
none,6929,4252,320
implicit,38,24,15
explicit,313,12,124


Layer A types behind implicit: {'gendered_title_pronoun': 38}
implicit distinct texts: ['booking salesman reliever', 'construction foreman', 'foreman', 'foreman (earthworks and civil)', 'foreman (earthworks/civil)', 'foreman civil', 'foreman electrical', 'foreman mechanical', 'foreman plumbing', 'general foreman', 'jr relief salesman', 'junior salesman', 'micro d salesman', 'saleslady', 'salesman']


## 2. Cross-validated detection and explicit/implicit classification

Pooled out-of-fold predictions. Evaluation helpers work from confusion matrices per employer, which makes the bootstrap exact and fast.

In [4]:
groups = ds.employer_id.values; ue, inv = np.unique(groups, return_inverse=True); G = len(ue)
folds = list(GroupKFold(n_splits=N_SPLITS).split(ds, ds.y3, groups))
rng = np.random.default_rng(SEED); W = rng.multinomial(G, np.ones(G) / G, size=N_BOOT)

def make(): return LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED)
def vec(): return TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2, sublinear_tf=True)

def oof(texts, y, frame, folds):
    pred = np.zeros(len(frame), int)
    for tr, te in folds:
        v = vec(); m = make().fit(v.fit_transform(texts[tr]), y[tr]); pred[te] = m.predict(v.transform(texts[te]))
    return pred

def cm_groups(y, p, K, inv, G):
    out = np.zeros((G, K, K)); np.add.at(out, (inv, y, p), 1); return out
def prf(cm):
    tp = np.diag(cm); p = np.divide(tp, cm.sum(0), out=np.zeros(len(cm)), where=cm.sum(0) > 0)
    r = np.divide(tp, cm.sum(1), out=np.zeros(len(cm)), where=cm.sum(1) > 0)
    f = np.divide(2 * p * r, p + r, out=np.zeros(len(cm)), where=(p + r) > 0); return p, r, f
def report(y, p, K, names, inv_, G_, W_):
    cmg = cm_groups(y, p, K, inv_, G_); cm = cmg.sum(0); P, R, F = prf(cm)
    bs = np.array([[*prf(np.tensordot(w, cmg, axes=(0, 0)))[2], prf(np.tensordot(w, cmg, axes=(0, 0)))[2].mean()] for w in W_])
    ci = lambda k: f"{np.percentile(bs[:, k], 2.5):.3f}-{np.percentile(bs[:, k], 97.5):.3f}"
    t = pd.DataFrame({"precision": P.round(3), "recall": R.round(3), "F1": F.round(3), "F1_95CI": [ci(k) for k in range(K)], "support": cm.sum(1).astype(int)}, index=names)
    t.loc["macro"] = [P.mean().round(3), R.mean().round(3), F.mean().round(3), ci(K), int(cm.sum())]
    return t, pd.DataFrame(cm.astype(int), index=[f"true_{n}" for n in names], columns=[f"pred_{n}" for n in names])

texts = ds.sentence.values
res = {}
p3 = oof(texts, ds.y3.values, ds, folds); pb = oof(texts, ds.yb.values, ds, folds)
res["3class"] = report(ds.y3.values, p3, 3, CL, inv, G, W); res["binary"] = report(ds.yb.values, pb, 2, ["not_biased", "biased"], inv, G, W)
print("=== Gender-bias detection (binary model) ==="); display(res["binary"][0]); display(res["binary"][1])
print("=== Explicit / implicit classification (3-class model) ==="); display(res["3class"][0]); display(res["3class"][1])
for k, (t, c) in res.items(): t.to_csv(OUT / f"step13_{k}_metrics.csv"); c.to_csv(OUT / f"step13_{k}_confusion.csv")
print("chance reference, always 'none': macro-F1 =", round(prf(np.array([[len(ds) - (ds.y3 > 0).sum(), 0, 0], [(ds.y3 == 1).sum(), 0, 0], [(ds.y3 == 2).sum(), 0, 0]]))[2].mean(), 3))
ds["pred3"] = [CL[i] for i in p3]; ds["predb"] = pb
ds[ds.pred3 != ds.bias_type][["sentence_id", "employer_id", "sentence", "bias_type", "pred3"]].to_csv(OUT / "step13_oof_errors.csv", index=False)
print("out-of-fold errors saved:", int((ds.pred3 != ds.bias_type).sum()))

=== Gender-bias detection (binary model) ===


,precision,recall,F1,F1_95CI,support
not_biased,0.998,1.000,0.999,0.998-1.000,6929
biased,1.000,0.957,0.978,0.954-0.995,351
macro,0.999,0.979,0.989,0.976-0.998,7280


,pred_not_biased,pred_biased
true_not_biased,6929,0
true_biased,15,336


=== Explicit / implicit classification (3-class model) ===


,precision,recall,F1,F1_95CI,support
none,0.999,1.000,0.999,0.999-1.000,6929
implicit,0.921,0.921,0.921,0.667-0.982,38
explicit,1.000,0.990,0.995,0.987-1.000,313
macro,0.973,0.970,0.972,0.889-0.992,7280


,pred_none,pred_implicit,pred_explicit
true_none,6926,3,0
true_implicit,3,35,0
true_explicit,3,0,310


chance reference, always 'none': macro-F1 = 0.325
out-of-fold errors saved: 9


## 3. Sensitivity runs

- **Keywords removed:** the gender words, titles, pronouns and inclusive phrases that define the labels are deleted before training and testing. This shows how much the models rely on the words that made the labels.
- **Distinct texts only:** one row per distinct sentence, so repeated postings cannot make the score look better.

In [5]:
def strip(s):
    for rx in (INCL_RE, TITLE_RE, PRON_BIN_RE, PRON_RE, re.compile(r"\b(?:%s)\b" % MALE["std"], re.I), re.compile(r"\b(?:%s)\b" % FEMALE["std"], re.I)):
        s = rx.sub(" ", s)
    return s
stripped = np.array([strip(s) for s in texts])
print("units changed by stripping:", int((stripped != texts).sum()), "of", len(texts))
sens = []
for name, tx in [("full text", texts), ("keywords removed", stripped)]:
    p3_, pb_ = (p3, pb) if name == "full text" else (oof(tx, ds.y3.values, ds, folds), oof(tx, ds.yb.values, ds, folds))
    t3 = report(ds.y3.values, p3_, 3, CL, inv, G, W)[0]; tb = report(ds.yb.values, pb_, 2, ["not_biased", "biased"], inv, G, W)[0]
    sens.append(dict(variant=name, n_units=len(ds), biased_P=tb.loc["biased", "precision"], biased_R=tb.loc["biased", "recall"], biased_F1=tb.loc["biased", "F1"],
                     F1_implicit=t3.loc["implicit", "F1"], F1_explicit=t3.loc["explicit", "F1"], macro_F1=t3.loc["macro", "F1"]))
# distinct texts
dd = ds.drop_duplicates("sentence").reset_index(drop=True); ued, invd = np.unique(dd.employer_id.values, return_inverse=True); Gd = len(ued)
Wd = np.random.default_rng(SEED).multinomial(Gd, np.ones(Gd) / Gd, size=N_BOOT); fd = list(GroupKFold(n_splits=N_SPLITS).split(dd, dd.y3, dd.employer_id.values))
q3 = oof(dd.sentence.values, dd.y3.values, dd, fd); qb = oof(dd.sentence.values, dd.yb.values, dd, fd)
t3 = report(dd.y3.values, q3, 3, CL, invd, Gd, Wd)[0]; tb = report(dd.yb.values, qb, 2, ["not_biased", "biased"], invd, Gd, Wd)[0]
sens.append(dict(variant="distinct texts only", n_units=len(dd), biased_P=tb.loc["biased", "precision"], biased_R=tb.loc["biased", "recall"], biased_F1=tb.loc["biased", "F1"],
                 F1_implicit=t3.loc["implicit", "F1"], F1_explicit=t3.loc["explicit", "F1"], macro_F1=t3.loc["macro", "F1"]))
sens = pd.DataFrame(sens).set_index("variant"); display(sens); sens.to_csv(OUT / "step13_sensitivity.csv")
print("distinct texts by class:", dd.bias_type.value_counts().to_dict())

units changed by stripping: 351 of 7280


,n_units,biased_P,biased_R,biased_F1,F1_implicit,F1_explicit,macro_F1
variant,,,,,,,
full text,7280,1.000,0.957,0.978,0.921,0.995,0.972
keywords removed,7280,0.378,0.954,0.542,0.107,0.557,0.542
distinct texts only,4288,1.000,0.917,0.957,0.933,1.000,0.978


distinct texts by class: {'none': 4252, 'implicit': 24, 'explicit': 12}


## 4. Span localisation

Token tagger trained only on gold-biased units of the training fold. Features: lower-cased word, 3-character prefix and suffix, capitalisation, position, and the two words either side. **No gender lexicon is given to the tagger**; it has to learn which words carry the bias from the data.

In [6]:
TOK = re.compile(r"\w+|[^\w\s]")
def toks(s): return [(m.start(), m.end(), m.group()) for m in TOK.finditer(s)]
def feats(ts, i):
    g = lambda j: ts[j][2].lower() if 0 <= j < len(ts) else "<pad>"
    w = ts[i][2]
    return {"w": g(i), "p3": g(i)[:3], "s3": g(i)[-3:], "title": str(w.istitle()), "upper": str(w.isupper()), "first": str(i == 0), "last": str(i == len(ts) - 1),
            "pw": g(i - 1), "nw": g(i + 1), "pw2": g(i - 2), "nw2": g(i + 2)}
def tag_data(sents, spans):
    X, y = [], []
    for s, sp in zip(sents, spans):
        ts = toks(s)
        for i, (a, b, _) in enumerate(ts):
            X.append(feats(ts, i)); y.append(int(sp is not None and a < sp[1] and b > sp[0]))
    return X, np.array(y)
def fit_tagger(sents, spans):
    X, y = tag_data(sents, spans); dv = DictVectorizer(); m = make().fit(dv.fit_transform(X), y); return dv, m
def predict_spans(tagger, s):
    dv, m = tagger; ts = toks(s)
    if not ts: return []
    pr = m.predict_proba(dv.transform([feats(ts, i) for i in range(len(ts))]))[:, list(m.classes_).index(1)] >= 0.5
    out, i = [], 0
    while i < len(ts):
        if pr[i]:
            j = i
            while j + 1 < len(ts) and pr[j + 1]: j += 1
            out.append((ts[i][0], ts[j][1])); i = j + 1
        else: i += 1
    return out
def iou(a, b):
    inter = max(0, min(a[1], b[1]) - max(a[0], b[0])); return inter / ((a[1] - a[0]) + (b[1] - b[0]) - inter)
def counts(s, pred, gold):
    ts = toks(s); ov = lambda sp: {i for i, (a, b, _) in enumerate(ts) if a < sp[1] and b > sp[0]}
    G_ = ov(gold) if gold else set(); P_ = set().union(*[ov(p) for p in pred]) if pred else set()
    ex = int(gold is not None and tuple(gold) in [tuple(p) for p in pred]); rl = int(gold is not None and any(iou(p, gold) >= 0.5 for p in pred))
    g1 = int(gold is not None)
    return [ex, len(pred) - ex, g1 - ex, rl, len(pred) - rl, g1 - rl, len(G_ & P_), len(P_ - G_), len(G_ - P_)]

gold_spans = [tuple(sp) if sp is not None else None for sp in ds.span]
cA = np.zeros((len(ds), 9)); cB = np.zeros((len(ds), 9)); predspans = [None] * len(ds)
for tr, te in folds:
    trb = [i for i in tr if ds.y3[i] > 0]
    tg = fit_tagger(ds.sentence.values[trb], [gold_spans[i] for i in trb])
    for i in te:
        ps = predict_spans(tg, ds.sentence[i]); predspans[i] = ps
        if ds.y3[i] > 0: cA[i] = counts(ds.sentence[i], ps, gold_spans[i])           # (a) given gold-biased unit
        cB[i] = counts(ds.sentence[i], ps if pb[i] == 1 else [], gold_spans[i] if ds.y3[i] > 0 else None)  # (b) end to end

def span_table(c, mask):
    cg = np.zeros((G, 9)); np.add.at(cg, inv[mask], c[mask]); tot = cg.sum(0); rows = {}
    def f(t): p = t[0] / (t[0] + t[1]) if t[0] + t[1] else 0; r = t[0] / (t[0] + t[2]) if t[0] + t[2] else 0; return p, r, (2 * p * r / (p + r) if p + r else 0)
    names = ["exact match", "overlap (IoU>=0.5)", "token overlap"]
    out = []
    for k, n in enumerate(names):
        P_, R_, F_ = f(tot[3 * k:3 * k + 3]); bs = np.array([f((w @ cg)[3 * k:3 * k + 3])[2] for w in W])
        out.append(dict(metric=n, precision=round(P_, 3), recall=round(R_, 3), F1=round(F_, 3), F1_95CI=f"{np.percentile(bs, 2.5):.3f}-{np.percentile(bs, 97.5):.3f}"))
    return pd.DataFrame(out).set_index("metric")
print("(a) Span localisation given a gold-biased unit  |  units:", int((ds.y3 > 0).sum())); sa = span_table(cA, (ds.y3 > 0).values); display(sa)
print("(b) End to end (tagger runs only where the detector says biased)  |  units:", len(ds)); sb = span_table(cB, np.ones(len(ds), bool)); display(sb)
sa.to_csv(OUT / "step13_span_given_biased.csv"); sb.to_csv(OUT / "step13_span_end_to_end.csv")
for cls in ("explicit", "implicit"):
    m = (ds.bias_type == cls).values; print(f"\n(a) {cls} only | units {int(m.sum())}"); display(span_table(cA, m))
bad = [(ds.sentence[i], gold_spans[i], predspans[i]) for i in range(len(ds)) if ds.y3[i] > 0 and cA[i, 0] == 0][:8]
print("\nExamples where the exact span was not matched (text, gold, predicted):"); [print(" ", b) for b in bad]

(a) Span localisation given a gold-biased unit  |  units: 351


,precision,recall,F1,F1_95CI
metric,,,,
exact match,1.0,0.994,0.997,0.993-1.000
overlap (IoU>=0.5),1.0,0.994,0.997,0.993-1.000
token overlap,1.0,0.994,0.997,0.993-1.000


(b) End to end (tagger runs only where the detector says biased)  |  units: 7280


,precision,recall,F1,F1_95CI
metric,,,,
exact match,1.0,0.954,0.977,0.951-0.995
overlap (IoU>=0.5),1.0,0.954,0.977,0.951-0.995
token overlap,1.0,0.954,0.977,0.951-0.995



(a) explicit only | units 313


...


(a) implicit only | units 38


,precision,recall,F1,F1_95CI
metric,,,,
exact match,1.0,0.947,0.973,0.917-1.000
overlap (IoU>=0.5),1.0,0.947,0.973,0.917-1.000
token overlap,1.0,0.947,0.973,0.917-1.000



Examples where the exact span was not matched (text, gold, predicted):


  ('Submit Reports Daily & Weekly for sites under he’s area of responsibility', (46, 48), [])
  ('Booking Salesman Reliever', (8, 16), [])


[None, None]

## 5. Final models, behaviour on the excluded `unclear` units, and a demo

Final models are fitted on all in-scope units and saved. The `unclear` units were never trained on, so the table below is the model's output on units that the schema leaves to a human. It is not scored.

In [7]:
v = vec(); Xall = v.fit_transform(ds.sentence.values)
m3 = make().fit(Xall, ds.y3.values); mb = make().fit(Xall, ds.yb.values)
bi = [i for i in range(len(ds)) if ds.y3[i] > 0]; tagger = fit_tagger(ds.sentence.values[bi], [gold_spans[i] for i in bi])
joblib.dump(dict(vectorizer=v, clf3=m3, clf_binary=mb, tagger=tagger, classes=CL, trained_on="PESO Step 8c rule silver labels (unclear excluded), not human validated"), OUT / "step13_gender_detector.joblib")

def detect(sents):
    X = v.transform(sents); c3 = m3.predict(X); b = mb.predict(X)
    return pd.DataFrame({"text": sents, "type": [CL[i] for i in c3], "biased": b.astype(bool), "span_text": [", ".join(s[a:e] for a, e in predict_spans(tagger, s)) if bb else "" for s, bb in zip(sents, b)]})
unc = d[d.bias_type == "unclear"]
t = pd.crosstab(unc.layerA, pd.Series([CL[i] for i in m3.predict(v.transform(unc.sentence.values))], index=unc.index, name="model type")); display(t); t.to_csv(OUT / "step13_unclear_units_model_output.csv")
demo = ["Male", "Female only", "Must be male, 25-30 years old", "Salesman needed", "Strong leadership and competitive spirit", "Female/Male",
        "With pleasing personality", "Production Associates (70)", "At least High School Graduate", "Must be willing to render overtime"]
display(detect(demo))

model type,explicit,none
layerA,,
binary_listing,101,5
gender_coded,0,577
unclear,0,8


,text,type,biased,span_text
0,Male,explicit,True,Male
1,Female only,explicit,True,Female
2,"Must be male, 25-30 years old",none,False,
3,Salesman needed,implicit,True,Salesman
4,Strong leadership and competitive spirit,none,False,
5,Female/Male,explicit,True,"Female, Male"
6,With pleasing personality,none,False,
7,Production Associates (70),none,False,
8,At least High School Graduate,none,False,
9,Must be willing to render overtime,none,False,


## 6. Reading guide and limits
- The labels were made by rules, so scores show reproduction of the rules. The keyword-removed run shows how much of the performance depends on the label-defining words.
- `implicit` here means only gendered titles and pronouns (rules never call coded wording bias), so it is a small class with few distinct texts; its F1 is unstable.
- `unclear` units are not scored. The schema leaves coded wording and "Female/Male" to a human.
- Gold spans come from the same rules, and the tagger sees the words (not the rule), so span scores are an upper bound on what a human-labelled test would show.
- No transformer was trained; this is a TF-IDF and logistic-regression baseline on CPU.
- Human validation on PESO is still pending.